# TP — Détecteur d'images générées par IA (anti-deepfake) — **TensorFlow / Keras**

**M2 DMIA — Deep Learning — Projet 4 (Session 3, ponts vers S7 et S8)**

## L'enjeu (2026)

En 2026, les images générées par IA sont partout : faux visuels d'actualité, fausses
photos de personnalités, désinformation pendant les élections. À l'œil nu, il devient
**très difficile** de distinguer une vraie photo d'une image synthétique. La question
de ce TP :

> **Une image est-elle une VRAIE photo, ou a-t-elle été GÉNÉRÉE PAR UNE IA ?**

Bonne nouvelle : les générateurs d'IA laissent souvent des **signatures** subtiles
(textures trop lisses, artefacts de fréquences, motifs de bruit). Un **CNN 2D** peut
apprendre à repérer ces signatures là où notre œil échoue.

## Ce que vous allez apprendre

- La **convolution 2D** : filtres, *feature maps*, *pooling* — le cœur de la vision par ordinateur.
- La **data augmentation** pour rendre le modèle plus robuste.
- Lire une **matrice de confusion** et analyser les erreurs.
- Des ouvertures concrètes : **transfer learning** (MobileNetV2, lien Session 7) et
  **explicabilité** (Grad-CAM, lien Session 8).

## Pipeline en 7 étapes (toujours le même !)

1. Charger les images → 2. Prétraiter → 3. Définir le modèle CNN → 4. Perte + optimiseur
→ 5. Entraîner → 6. Évaluer → 7. Analyser + Bonus.

> **Recommandé** : exécuter sur **Google Colab** avec **GPU** (Exécution → Modifier le type
> d'exécution → GPU). Le notebook tourne aussi sur CPU grâce à un jeu de données réduit.

> **Robustesse** : on tente de télécharger le **vrai** dataset CIFAKE (vraies vs fausses
> images). En cas d'échec, un **repli synthétique automatique** construit à partir de
> CIFAR-10 (intégré à Keras) garantit que le TP **tourne toujours**, même hors-ligne.

## 1. Imports et reproductibilité

On importe les outils habituels. **TensorFlow / Keras** pour le réseau, **NumPy** pour les
tableaux, **Matplotlib** pour les graphiques, **scikit-learn** pour la matrice de confusion.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproductibilite : meme graine (seed) -> memes resultats
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print('TensorFlow version :', tf.__version__)

## 2. Charger les images (réel + repli automatique)

Stratégie en deux temps :

1. **Données RÉELLES** : on tente de télécharger **CIFAKE**
   (`birdy654/cifake-real-and-ai-generated-synthetic-images`) via `kagglehub`. Ce dataset
   contient ~60 000 images réelles (CIFAR-10) et ~60 000 images **réellement générées par IA**
   (modèle de diffusion), en 32×32. On charge les dossiers `train/REAL` et `train/FAKE` avec
   `image_dataset_from_directory`.

2. **Repli SYNTHÉTIQUE** (si le téléchargement échoue) : on construit un jeu binaire à partir
   de **CIFAR-10** :
   - classe **RÉELLE** = images CIFAR-10 telles quelles ;
   - classe **FAUSSE** = mêmes images **dégradées** (flou gaussien + bruit structuré + léger
     sur-lissage) pour **simuler** des artefacts d'IA.

> ⚠️ **Honnêteté pédagogique** : le repli n'utilise **pas de vraies images IA**. C'est une
> **simulation** du problème, juste pour faire tourner le **pipeline CNN** de bout en bout.
> Un vrai détecteur d'images IA s'entraîne sur de vraies images générées (comme CIFAKE).

In [ ]:
# Limite pour que ca tourne vite (sur CPU aussi)
MAX_IMAGES = 10000   # ~5000 reelles + ~5000 fausses
IMG_SIZE = 32        # CIFAKE et CIFAR-10 sont en 32x32
SOURCE = None        # sera "CIFAKE" ou "SYNTHETIQUE"


def degrader_images_ia(images):
    '''Simule des artefacts d'images IA : flou gaussien + bruit structure + sur-lissage.
    Entree/sortie : tableau float32 dans [0, 1], forme (N, H, W, 3).'''
    imgs = images.astype('float32')

    # 1) Flou gaussien leger via une moyenne 3x3 (sur-lissage des textures)
    noyau = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype='float32')
    noyau = noyau / noyau.sum()
    flou = np.zeros_like(imgs)
    pad = np.pad(imgs, ((0, 0), (1, 1), (1, 1), (0, 0)), mode='reflect')
    for di in range(3):
        for dj in range(3):
            flou += noyau[di, dj] * pad[:, di:di + IMG_SIZE, dj:dj + IMG_SIZE, :]

    # 2) Bruit structure (motif periodique faible, typique de certains generateurs)
    yy, xx = np.meshgrid(np.arange(IMG_SIZE), np.arange(IMG_SIZE), indexing='ij')
    motif = 0.04 * np.sin(2 * np.pi * xx / 4.0) * np.cos(2 * np.pi * yy / 4.0)
    motif = motif[None, :, :, None]

    # 3) Leger sur-lissage supplementaire (melange image floue + originale)
    sortie = 0.7 * flou + 0.3 * imgs + motif
    return np.clip(sortie, 0.0, 1.0).astype('float32')


def charger_donnees():
    '''Retourne (X, y, source). X dans [0,1], y dans {0=REELLE, 1=FAUSSE}.'''
    # --- Tentative 1 : vrai dataset CIFAKE via kagglehub ---
    try:
        import kagglehub
        from tensorflow.keras.utils import image_dataset_from_directory
        import os

        path = kagglehub.dataset_download(
            'birdy654/cifake-real-and-ai-generated-synthetic-images')
        train_dir = os.path.join(path, 'train')
        ds = image_dataset_from_directory(
            train_dir,
            labels='inferred',
            label_mode='int',           # FAKE / REAL -> 0 / 1 (ordre alphabetique)
            image_size=(IMG_SIZE, IMG_SIZE),
            batch_size=256,
            shuffle=True,
            seed=SEED,
        )
        # noms de classes : ['FAKE', 'REAL'] -> on veut 0=REELLE, 1=FAUSSE
        class_names = ds.class_names
        idx_fake = class_names.index('FAKE')

        X_list, y_list = [], []
        n = 0
        for batch_x, batch_y in ds:
            X_list.append(batch_x.numpy())
            # y=1 si FAKE (fausse), 0 sinon (reelle)
            y_list.append((batch_y.numpy() == idx_fake).astype('int32'))
            n += batch_x.shape[0]
            if n >= MAX_IMAGES:
                break
        X = np.concatenate(X_list)[:MAX_IMAGES] / 255.0
        y = np.concatenate(y_list)[:MAX_IMAGES]
        return X.astype('float32'), y.astype('int32'), 'CIFAKE'

    except Exception as e:
        print('[INFO] CIFAKE indisponible (', type(e).__name__, ') -> repli synthetique.')

    # --- Tentative 2 : repli synthetique a partir de CIFAR-10 ---
    (x_cifar, _), _ = keras.datasets.cifar10.load_data()
    n_par_classe = MAX_IMAGES // 2
    reelles = x_cifar[:n_par_classe].astype('float32') / 255.0
    base_fausses = x_cifar[n_par_classe:2 * n_par_classe].astype('float32') / 255.0
    fausses = degrader_images_ia(base_fausses)

    X = np.concatenate([reelles, fausses], axis=0).astype('float32')
    y = np.concatenate([
        np.zeros(len(reelles), dtype='int32'),   # 0 = REELLE
        np.ones(len(fausses), dtype='int32'),    # 1 = FAUSSE
    ])
    # melange
    perm = np.random.permutation(len(X))
    return X[perm], y[perm], 'SYNTHETIQUE'


X, y, SOURCE = charger_donnees()

if SOURCE == 'CIFAKE':
    print('Donnees REELLES (CIFAKE)')
else:
    print('Repli SYNTHETIQUE (CIFAR-10 degrade)')

print('X :', X.shape, '| y :', y.shape)
print('Pixels : min =', round(float(X.min()), 3), ', max =', round(float(X.max()), 3))
print('Repartition : REELLES =', int((y == 0).sum()), '| FAUSSES =', int((y == 1).sum()))

### Afficher une grille d'exemples : RÉELLE vs FAUSSE

Regardons quelques images de chaque classe. Sur le repli synthétique, on devine que les
« fausses » sont **légèrement plus lisses / texturées différemment**. Sur CIFAKE, la
distinction est beaucoup plus subtile — c'est tout l'intérêt du CNN.

In [ ]:
classes = ['REELLE', 'FAUSSE']

def montrer_grille(X, y, titre):
    plt.figure(figsize=(11, 3))
    idx_reelles = np.where(y == 0)[0][:6]
    idx_fausses = np.where(y == 1)[0][:6]
    for i, idx in enumerate(idx_reelles):
        plt.subplot(2, 6, i + 1)
        plt.imshow(np.clip(X[idx], 0, 1))
        if i == 0:
            plt.ylabel('REELLE', fontsize=11)
        plt.xticks([]); plt.yticks([])
    for i, idx in enumerate(idx_fausses):
        plt.subplot(2, 6, 6 + i + 1)
        plt.imshow(np.clip(X[idx], 0, 1))
        if i == 0:
            plt.ylabel('FAUSSE', fontsize=11)
        plt.xticks([]); plt.yticks([])
    plt.suptitle(titre)
    plt.tight_layout(); plt.show()

montrer_grille(X, y, 'Haut = REELLES, Bas = FAUSSES (source : ' + SOURCE + ')')

## 3. Prétraiter : normalisation + split train / test

- **Normalisation** : les pixels sont déjà ramenés dans **[0, 1]** au chargement
  (l'optimisation converge mieux quand les entrées sont petites et centrées).
- **Split train / test** : on réserve **20 %** des images pour le **test final** (jamais vues
  pendant l'entraînement), afin de mesurer la **généralisation**. On stratifie pour garder la
  même proportion de classes des deux côtés.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print('Train :', X_train.shape, '| Test :', X_test.shape)
print('Train  -> REELLES =', int((y_train == 0).sum()), '| FAUSSES =', int((y_train == 1).sum()))
print('Test   -> REELLES =', int((y_test == 0).sum()),  '| FAUSSES =', int((y_test == 1).sum()))

## 4. Définir le modèle CNN 2D

C'est le **CNN 2D par excellence**. Idée clé : au lieu d'aplatir l'image et de tout connecter
(comme un MLP), un CNN fait **glisser de petits filtres** (3×3) sur l'image pour détecter des
**motifs locaux** (bords, textures, artefacts). Chaque filtre produit une **feature map**.

Notre architecture :

```
Data augmentation (RandomFlip + RandomRotation)   <- robustesse
Conv2D(32, 3x3, relu) + BatchNorm + MaxPool       <- motifs simples (bords)
Conv2D(64, 3x3, relu) + BatchNorm + MaxPool       <- motifs plus riches (textures)
Conv2D(64, 3x3, relu) + MaxPool                   <- motifs abstraits
Flatten -> Dense(64, relu) + Dropout(0.5)         <- decision
Dense(1, sigmoid)                                 <- proba "FAUSSE"
```

- **BatchNorm** : stabilise et accélère l'entraînement.
- **MaxPooling** : réduit la taille (résume chaque zone par son max) → moins de calcul, invariance.
- **Dropout** : éteint 50 % des neurones au hasard pendant l'entraînement → limite le sur-apprentissage.
- **Sortie sigmoid** (1 neurone) : probabilité que l'image soit **FAUSSE** (classification **binaire**).

La **data augmentation** est intégrée **au début du modèle** : à chaque époque, les images sont
légèrement retournées / pivotées. Elle ne s'active **qu'à l'entraînement** (automatique avec Keras).

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
], name='data_augmentation')

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),   # proba "FAUSSE"
], name='cnn_anti_deepfake')

model.summary()

### Perte + optimiseur

- **Optimiseur** : Adam (un bon défaut).
- **Perte** : `binary_crossentropy` (classification binaire, sortie sigmoid).
- **Métrique** : accuracy.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

## 5. Entraîner

On entraîne sur quelques époques avec **10 %** des données d'entraînement en **validation**
pour surveiller le sur-apprentissage. Sur GPU c'est rapide ; sur CPU comptez quelques minutes.

In [ ]:
EPOCHS = 12

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=128,
    verbose=1,
)

### Courbes d'apprentissage : train vs validation

On compare la **perte** et l'**accuracy** en entraînement et en validation. Si la courbe de
validation décroche de celle d'entraînement, c'est le signe d'un **sur-apprentissage**.

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Perte'); plt.xlabel('epoque'); plt.ylabel('loss'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy'); plt.xlabel('epoque'); plt.ylabel('accuracy'); plt.legend()

plt.tight_layout(); plt.show()

## 6. Évaluer : accuracy test + matrice de confusion

On mesure l'accuracy sur le **jeu de test** (images jamais vues), puis on regarde la
**matrice de confusion** pour comprendre **quelles** erreurs le modèle commet :
- vraies positives / négatives = bonnes décisions ;
- fausses alertes (une vraie photo classée FAUSSE) vs ratés (une fausse classée RÉELLE).

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy sur le test : {test_acc:.4f}  (perte : {test_loss:.4f})')

# Probabilites -> classes (seuil 0.5)
proba = model.predict(X_test, verbose=0).ravel()
y_pred = (proba >= 0.5).astype('int32')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap='Blues', colorbar=False)
plt.title('Matrice de confusion (test)')
plt.show()

print(classification_report(y_test, y_pred, target_names=classes))

### Quelques images mal classées

Visualiser les erreurs aide à comprendre les **limites** du modèle (images ambiguës,
artefacts trop discrets...). Titre : `v:` = vraie classe, `p:` = prédiction.

In [ ]:
erreurs = np.where(y_pred != y_test)[0]
print('Nombre d\'erreurs :', len(erreurs), 'sur', len(y_test))

n_show = min(10, len(erreurs))
if n_show > 0:
    plt.figure(figsize=(12, 2.5))
    for i, idx in enumerate(erreurs[:n_show]):
        plt.subplot(1, n_show, i + 1)
        plt.imshow(np.clip(X_test[idx], 0, 1))
        plt.title('v:' + classes[y_test[idx]] + '\np:' + classes[y_pred[idx]], fontsize=8)
        plt.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Aucune erreur a afficher (rare, mais possible).')

## 7. BONUS — Ouvertures vers la suite du cours

### (a) Transfer learning avec MobileNetV2 (lien Session 7)

Plutôt que d'entraîner un CNN **de zéro**, on peut réutiliser un réseau **pré-entraîné** sur
des millions d'images (ImageNet) comme **extracteur de features**. On « gèle » ses poids et on
ne ré-entraîne qu'une petite tête de classification. C'est souvent **plus précis** et **plus
rapide**, surtout avec peu de données — le cœur de la **Session 7**.

> MobileNetV2 attend des entrées d'au moins 32×32 ; il intègre son propre prétraitement
> (`preprocess_input`, qui ramène les pixels dans [-1, 1]). Code minimal ci-dessous,
> **optionnel** (décommentez `fit` pour l'entraîner).

In [ ]:
# --- Ebauche de transfer learning (optionnel) ---
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,        # on enleve la tete de classification ImageNet
    weights='imagenet',
)
base.trainable = False        # on gele l'extracteur de features

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = layers.Rescaling(255.0)(inputs)          # nos pixels sont en [0,1] -> repasser en [0,255]
x = preprocess_input(x)                      # prepro specifique MobileNetV2 -> [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model_tl = keras.Model(inputs, outputs, name='transfer_mobilenetv2')
model_tl.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_tl.summary()

# Decommentez pour entrainer la tete (seuls les poids de la tete bougent) :
# history_tl = model_tl.fit(X_train, y_train, validation_split=0.1, epochs=5, batch_size=128)
# print('Accuracy test (transfer) :', model_tl.evaluate(X_test, y_test, verbose=0)[1])

### (b) Explicabilité avec Grad-CAM (lien Session 8)

Un détecteur d'images IA ne doit pas être une **boîte noire** : pour la confiance et l'éthique
(**Session 8**), il faut pouvoir montrer **où** le réseau regarde. **Grad-CAM** produit une
**carte de chaleur** sur l'image : il calcule le gradient de la sortie par rapport à la
**dernière couche convolutive**, et met en évidence les zones qui ont **le plus pesé** dans la
décision.

L'intuition : si le modèle dit « FAUSSE », Grad-CAM révèle quelles **régions / textures** l'ont
trahi. Code court ci-dessous (optionnel) qui calcule et superpose la carte de chaleur sur une
image de test.

In [ ]:
# --- Grad-CAM (optionnel) ---
def grad_cam(model, image, last_conv_name=None):
    '''Carte de chaleur Grad-CAM pour une image (H, W, 3) dans [0,1].'''
    # trouver la derniere couche Conv2D si non precisee
    if last_conv_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, layers.Conv2D):
                last_conv_name = layer.name
                break

    grad_model = keras.Model(
        model.inputs,
        [model.get_layer(last_conv_name).output, model.output])

    img = image[None, ...].astype('float32')
    with tf.GradientTape() as tape:
        conv_out, pred = grad_model(img)
        classe = pred[:, 0]            # sortie sigmoid "FAUSSE"

    grads = tape.gradient(classe, conv_out)[0]          # (h, w, c)
    poids = tf.reduce_mean(grads, axis=(0, 1))          # importance de chaque canal
    heat = tf.reduce_sum(conv_out[0] * poids, axis=-1)  # (h, w)
    heat = tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)
    return heat.numpy()


# Exemple sur la premiere image de test
ex = X_test[0]
heat = grad_cam(model, ex)

plt.figure(figsize=(8, 3))
plt.subplot(1, 3, 1); plt.imshow(np.clip(ex, 0, 1)); plt.title('Image'); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(heat, cmap='jet'); plt.title('Grad-CAM'); plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(np.clip(ex, 0, 1))
heat_resized = np.array(tf.image.resize(heat[..., None], (IMG_SIZE, IMG_SIZE)))[..., 0]
plt.imshow(heat_resized, cmap='jet', alpha=0.5)
plt.title('Superposition'); plt.axis('off')
plt.tight_layout(); plt.show()

## 8. À toi de jouer

Expérimente et note l'effet sur l'**accuracy de test** :

1. **Plus profond** : ajoute un 4ᵉ bloc `Conv2D(128) + BatchNorm + MaxPool`. Gain ?
2. **Augmentation** : ajoute `layers.RandomZoom(0.1)` ou `layers.RandomContrast(0.1)` dans
   `data_augmentation`. La validation s'améliore-t-elle ?
3. **Régularisation** : fais varier le `Dropout` (0.3, 0.5, 0.7). Effet sur le sur-apprentissage ?
4. **Transfer learning** : entraîne réellement `model_tl` (décommente le `fit`). Bat-il le CNN maison ?
5. **Fine-tuning** : après avoir entraîné la tête, dégèle les dernières couches de MobileNetV2
   (`base.trainable = True` + faible learning rate) et ré-entraîne.

### Discussion : les limites (important !)

- **Généralisation à de NOUVEAUX générateurs** : un modèle entraîné sur les images d'un
  générateur (ex. un modèle de diffusion de 2024) **généralise mal** à un générateur **inconnu**
  (sorti en 2026). Les signatures changent — c'est une **course à l'armement** permanente.
- **Repli synthétique** : si tu as utilisé le repli, rappelle-toi que les « fausses » images
  sont **simulées** (flou + bruit), pas de vraies images IA. Les résultats ne reflètent **pas**
  la difficulté réelle du problème (sur CIFAKE, c'est bien plus dur).
- **Éthique (S8)** : un détecteur imparfait peut produire des **fausses accusations** (vraie
  photo classée « IA »). D'où l'importance de l'**explicabilité** (Grad-CAM) et de la prudence
  avant toute décision automatique.

> Note tes résultats dans un petit tableau (config → accuracy test) et compare CNN maison vs transfer learning.

In [ ]:
# Ton espace d'experimentation
# ...
